In [1]:
import os
import platform
import sys

print(sys.version)
print(platform.platform())

import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Linux-6.6.122+-x86_64-with-glibc2.35
CUDA: True
GPU: Tesla T4


In [2]:
!nvidia-smi

Sun Aug  2 16:29:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import time
import numpy as np

def bench_matmul(n, iters=3):
    """测试 n×n 矩阵乘法, 返回平均用时和 GFLOPS"""
    times = []
    for _ in range(iters):
        A = np.random.rand(n, n)
        B = np.random.rand(n, n)
        t0 = time.perf_counter()
        C = A @ B
        times.append(time.perf_counter() - t0)
    avg = np.mean(times)
    gflops = 2 * n**3 / avg / 1e9   # 矩阵乘法 FLOPs = 2*n^3
    return avg, gflops

def bench_ops(n, iters=5):
    """测试向量运算: 点积 / 范数 / 求和"""
    a, b = np.random.rand(n), np.random.rand(n)
    res = {}
    for name, fn in [
        ("dot",  lambda: np.dot(a, b)),
        ("norm", lambda: np.linalg.norm(a)),
        ("sum",  lambda: a.sum()),
    ]:
        t0 = time.perf_counter()
        for _ in range(iters):
            fn()
        res[name] = (time.perf_counter() - t0) / iters * 1000  # ms
    return res

print(f"NumPy {np.__version__}")
print("\n=== Matrix Multiplication (GFLOPS) ===")
for n in [256, 512, 1024, 2048]:
    avg, gf = bench_matmul(n)
    print(f"{n:>6}x{n:<6}  {avg*1000:8.2f} ms   {gf:8.2f} GFLOPS")

print("\n=== Vector Operations (ms) ===")
for n in [10_000, 100_000, 1_000_000]:
    r = bench_ops(n)
    print(f"{n:>10,}  dot={r['dot']:.4f}  norm={r['norm']:.4f}  sum={r['sum']:.4f}")

NumPy 2.0.2

=== Matrix Multiplication (GFLOPS) ===
   256x256         1.19 ms      28.12 GFLOPS
   512x512         8.02 ms      33.46 GFLOPS
  1024x1024       62.71 ms      34.24 GFLOPS
  2048x2048      488.27 ms      35.19 GFLOPS

=== Vector Operations (ms) ===
    10,000  dot=0.0080  norm=0.0121  sum=0.0122
   100,000  dot=0.0733  norm=0.0317  sum=0.0601
 1,000,000  dot=0.8981  norm=0.3434  sum=0.5561
